# Talking to Ollama from Python — three interfaces

All three of these hit the *same* local daemon (`ollama serve`, listening on
`http://127.0.0.1:11434` by default). They differ only in how much sits between your
code and that HTTP endpoint:

| | Interface | Why you'd pick it |
|---|---|---|
| **1** | `ollama` — the official Python package | Ollama-native features: model management, `keep_alive`, `think`, raw `options` |
| **2** | `openai` SDK against `/v1` | Portability — identical code runs against a hosted API later |
| **3** | Plain `requests` / `httpx` | Zero extra dependencies, full control, 1:1 with the `curl` docs |

Model throughout: **`deepseek-r1`**, a *reasoning* model. That matters — it emits a
chain of thought before its answer, and each interface exposes that differently.
Section 4 covers the reasoning-specific gotchas.

## 0. Prerequisites

In a terminal:

```bash
ollama --version          # confirm the daemon is installed
ollama pull deepseek-r1   # ~5 GB for the default 8B tag; also :1.5b, :7b, :14b, :32b, :70b
ollama list               # confirm it landed
```

`ollama serve` runs automatically as a background service on macOS once the app has been
launched once. If nothing is listening, start it by hand with `ollama serve`.

In [1]:
# Run once. --quiet keeps the notebook output tidy.
%pip install --quiet --upgrade ollama openai requests pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.41 requires requests_mock, which is not installed.
streamlit 1.53.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.0 which is incompatible.
conda-repo-cli 1.0.41 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.41 requires nbformat==5.4.0, but you have nbformat 5.10.4 which is incompatible.
conda-repo-cli 1.0.41 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.41 requires PyYAML==6.0, but you have pyyaml 6.0.2 which is incompatible.
conda-repo-cli 1.0.41 requires requests==2.28.1, but you have requests 2.34.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to

In [2]:
import json, re, time, textwrap

HOST  = "http://127.0.0.1:11434"
MODEL = "deepseek-r1"        # or "deepseek-r1:1.5b" if you want something that fits in RAM easily

def wrap(text, width=88):
    '''Print long model output without a 400-character-wide line.'''
    for para in str(text).split("\n"):
        print(textwrap.fill(para, width) if para.strip() else "")

In [5]:
# Health check: is the daemon up, and is the model actually pulled?
import requests

try:
    ver = requests.get(f"{HOST}/api/version", timeout=3).json()
    print("ollama version:", ver["version"])
except requests.exceptions.ConnectionError:
    raise SystemExit(f"Nothing listening on {HOST} — start the daemon with `ollama serve`.")

tags = requests.get(f"{HOST}/api/tags", timeout=10).json()["models"]
names = [m["name"] for m in tags]
print("\ninstalled models:")
for n in names:
    print("  ", n)

if not any(n.split(":")[0] == MODEL.split(":")[0] for n in names):
    print(f"\n!! {MODEL} is not pulled. Run:  ollama pull {MODEL}")

ollama version: 0.33.2

installed models:
   deepseek-r1:latest
   llama3.1:8b


---
# 1. The official `ollama` package

`pip install ollama`. A thin, typed wrapper over the REST API — the response objects are
Pydantic models, so `resp.message.content` and `resp["message"]["content"]` both work.

The module-level functions (`chat`, `generate`, `list`, ...) are just a lazily-created
default `Client()` pointed at `OLLAMA_HOST` or `localhost:11434`.

### 1.1 Basic chat, with reasoning split out

Pass `think=True` and the daemon separates the chain of thought into its own
`message.thinking` field instead of leaving `<think>...</think>` tags inside
`message.content`. This is the single most useful thing to know when working with R1:
it means your downstream parsing never has to see the reasoning.

In [6]:
from ollama import chat

resp = chat(
    model=MODEL,
    messages=[{"role": "user",
               "content": "A bat and a ball cost $1.10 in total. "
                          "The bat costs $1.00 more than the ball. "
                          "How much does the ball cost?"}],
    think=True,                       # -> reasoning lands in message.thinking
    options={"temperature": 0.6},     # DeepSeek's own recommendation for R1
)

print("=== THINKING " + "=" * 60)
wrap(resp.message.thinking)
print("\n=== ANSWER " + "=" * 62)
wrap(resp.message.content)

print("\n=== METRICS " + "=" * 61)
print(f"prompt tokens : {resp.prompt_eval_count}")
print(f"output tokens : {resp.eval_count}   (reasoning tokens are counted here too)")
print(f"wall time     : {resp.total_duration / 1e9:.1f} s")
print(f"gen rate      : {resp.eval_count / (resp.eval_duration / 1e9):.1f} tok/s")

ImportError: cannot import name 'Sentinel' from 'typing_extensions' (/Users/breann/anaconda3/lib/python3.11/site-packages/typing_extensions.py)

### 1.2 Fallback: splitting `<think>` tags yourself

If you're on an older Ollama build, talking to a model the daemon doesn't recognise as a
thinking model, or using an interface that has no `think` flag (§2 and parts of §3), the
reasoning arrives inline wrapped in `<think>...</think>`. 

### 1.3 Streaming

`stream=True` turns the call into a generator of partial responses. With `think=True`
the thinking tokens stream first (in `chunk.message.thinking`), then the answer tokens
(in `chunk.message.content`) — so you can render a collapsible "thinking..." spinner and
only start showing the real answer when `content` becomes non-empty.

In [ ]:
import sys

stream = chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain why the sky is blue in two sentences."}],
    stream=True,
    think=True,
)

phase = None
for chunk in stream:
    if chunk.message.thinking:
        if phase != "think":
            print("\n--- thinking ---"); phase = "think"
        sys.stdout.write(chunk.message.thinking)
    if chunk.message.content:
        if phase != "answer":
            print("\n\n--- answer ---"); phase = "answer"
        sys.stdout.write(chunk.message.content)
    sys.stdout.flush()
print()

### 1.4 Multi-turn conversation

There is no server-side session — *you* own the history. Append each response's
`message` back onto the list before the next turn.

Important for R1: append the **answer only**, not the thinking. DeepSeek's guidance is
to exclude prior chain-of-thought from the context on subsequent turns; feeding it back
degrades quality and burns context for nothing.

In [ ]:
history = []

def ask(question: str, show_thinking: bool = False) -> str:
    history.append({"role": "user", "content": question})
    r = chat(model=MODEL, messages=history, think=True, options={"temperature": 0.6})
    if show_thinking:
        print("[thinking]", (r.message.thinking or "")[:300], "...\n")
    # NB: content only — deliberately dropping r.message.thinking from the history
    history.append({"role": "assistant", "content": r.message.content})
    return r.message.content


wrap(ask("I have a Mac with 16 GB of RAM. Which deepseek-r1 tag should I run?"))
print("\n" + "-" * 88 + "\n")
wrap(ask("And how much slower would the next size up be?"))
print(history)

### 1.5 Async

`AsyncClient` mirrors `Client` with awaitable methods. Note that in Jupyter you `await`
directly at cell level — `asyncio.run()` will raise because ipykernel already has a
running event loop.

In [ ]:
import asyncio
from ollama import AsyncClient

aclient = AsyncClient(host=HOST)

async def ask_async(q):
    r = await aclient.chat(model=MODEL, messages=[{"role": "user", "content": q}], think=True)
    return q, r.message.content.strip()

questions = [
    "In one sentence: what is a race condition?",
    "In one sentence: what is a deadlock?",
    "In one sentence: what is livelock?",
]

t0 = time.time()
results = await asyncio.gather(*(ask_async(q) for q in questions))   # top-level await
print(f"three questions in {time.time() - t0:.1f} s\n")

for q, a in results:
    print("Q:", q)
    wrap("A: " + a)
    print()

> **Caveat on concurrency.** Parallelism is bounded by the daemon, not by your client.
> `OLLAMA_NUM_PARALLEL` (default is chosen automatically from available memory) sets how
> many requests one loaded model serves concurrently; beyond that they queue. With a
> single 8B model on a laptop you're mostly sharing one GPU, so `gather` reduces
> round-trip overhead rather than multiplying throughput.

### 1.6 Structured output

Pass a JSON Schema as `format` and the daemon constrains decoding so the output *must*
parse against it. Pydantic's `model_json_schema()` gives you the schema for free.

With a reasoning model this only works cleanly in combination with `think=True`: the
constraint applies to `content`, so the model needs somewhere else to put its reasoning.
Without it, the grammar forbids the opening `<think>` token and you get either a refusal
to reason or malformed output.

In [ ]:
from pydantic import BaseModel, Field

class CodeReview(BaseModel):
    summary: str = Field(description="One sentence on what the function does")
    bugs: list[str] = Field(description="Concrete defects, empty list if none")
    severity: str = Field(description="one of: none, low, medium, high")

SNIPPET = '''
def mean(xs):
    total = 0
    for x in xs:
        total += x
    return total / len(xs)
'''

r = chat(
    model=MODEL,
    messages=[{"role": "user",
               "content": f"Review this Python function.\n```python{SNIPPET}```"}],
    format=CodeReview.model_json_schema(),
    think=True,
    options={"temperature": 0},      # determinism matters more than flair here
)

review = CodeReview.model_validate_json(r.message.content)
print(review.model_dump_json(indent=2))

### 1.7 Model management and embeddings

Everything the CLI does is on the client too — handy for scripts that need to guarantee a
model is present before they start work.

In [ ]:
import ollama

# What's loaded in memory right now, and when will it be evicted?
for m in ollama.ps().models:
    print(f"{m.name:24s} {m.size / 1e9:5.1f} GB  expires {m.expires_at:%H:%M:%S}")

print()
info = ollama.show(MODEL)
print("family      :", info.details.family)
print("parameters  :", info.details.parameter_size)
print("quantisation:", info.details.quantization_level)
print("context len :", info.modelinfo.get("general.architecture"),
      info.modelinfo.get(f"{info.details.family}.context_length"))

### 1.8 Runtime options worth knowing

`options` maps straight onto the modelfile parameters. The ones that actually bite:

| Option | Effect |
|---|---|
| `num_ctx` | Context window in tokens. **Defaults to 4096 regardless of what the model supports** — R1 will silently truncate long prompts unless you raise it. |
| `num_predict` | Hard cap on output tokens. Essential for R1, which can otherwise think for thousands of tokens. `-1` = unlimited. |
| `temperature` | DeepSeek recommend 0.6 for R1 (0 makes it loop, >0.8 makes it wander). |
| `top_p`, `top_k` | 0.95 / 40 are DeepSeek's suggested pairing with temp 0.6. |
| `seed` | Fixes sampling for reproducibility — pair with `temperature=0`. |
| `stop` | List of stop strings. |

`keep_alive` is a top-level argument, not an option: how long the model stays resident
after the call. `"30m"`, `0` to unload immediately, `-1` to pin it.

In [ ]:
r = chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Give me a haiku about a segfault."}],
    think=True,
    options={
        "num_ctx": 16384,       # raise from the 4096 default
        "num_predict": 512,     # stop R1 from thinking indefinitely
        "temperature": .6,
        "top_p": 0.95,
        "seed": 42,
    },
    keep_alive="30m",           # keep it warm for the rest of this notebook
)
wrap(r.message.content)
print(f"\n(stopped because: {r.done_reason})")

---
# 2. The OpenAI-compatible endpoint

Ollama serves an OpenAI-style API at `/v1`. Point the official `openai` SDK at it and
other libraries built on that SDK, such as LangChain, LlamaIndex, Instructor, most agent
frameworks, will work against your local model with a two-line change.

The API key is required by the SDK but ignored by the server; any non-empty string works.

### 2.1 Basic interface

In [ ]:
from openai import OpenAI

oai = OpenAI(base_url=f"{HOST}/v1", api_key="ollama")   # key is a placeholder

r = oai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is 17 * 23? Answer with the number only."}],
    temperature=0.6,
)

msg = r.choices[0].message
print("reasoning:", msg.model_extra['reasoning'], "...\n")
print("answer   :", msg.content)
print("\nusage:", r.usage.prompt_tokens, "in /", r.usage.completion_tokens, "out")

### 2.2 Streaming output

In [ ]:
def stream_answer(client, model, prompt, max_tokens=1024):
    stream = client.chat.completions.create(
        model=model, messages=[{'role': 'user', 'content': prompt}],
        stream=True, max_tokens=max_tokens)

    thinking, answer, phase = [], [], None
    for chunk in stream:
        if not chunk.choices:                      # usage-only final chunk
            continue
        d = chunk.choices[0].delta
        r = (d.model_extra or {}).get('reasoning_content') or ''
        c = d.content or ''

        if r:
            if phase != 'think':
                print('\n--- thinking ---', flush=True); 
                phase = 'think'
            thinking.append(r); print(r, end='', flush=True)
        if c:
            if phase != 'answer':
                print('\n--- answer ---', flush=True); 
                phase = 'answer'
            answer.append(c); print(c, end='', flush=True)

    print()
    return ''.join(thinking), ''.join(answer)

stream_answer(oai, MODEL, 'List three causes of memory fragmentation.')

In [ ]:
# Streaming, OpenAI-style: chunk.choices[0].delta.content
stream = oai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "List three causes of memory fragmentation."}],
    stream=True,
    max_tokens=600,
)

thinking_buf = []
ans_buf = []
for chunk in stream:
    d= chunk.choices[0].delta
    try:
        tok = d.reasoning
        thinking_buf.append(tok)
    except:
        tok = chunk.choices[0].delta.content
        ans_buf.append(tok)
        break
    if tok is not None:
        sys.stdout.write(tok)
        sys.stdout.flush()
print("\n")

wrap("Answer with reasoning stripped:\n\n" + ''.join(ans_buf))

### Structured output

In [ ]:
# Structured output, OpenAI-style. Ollama maps response_format onto the same
# grammar-constrained decoding as format= in section 1.6.
schema = CodeReview.model_json_schema()
schema["additionalProperties"] = False          # strict mode requires this

r = oai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user",
               "content": f"Review this function and reply as JSON.\n```python{SNIPPET}```"}],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "code_review", "schema": schema, "strict": True},
    },
    temperature=0,
)

print(json.dumps(json.loads(r.choices[0].message.content), indent=2))

> **Portability demo.** The only thing distinguishing local from hosted is the two
> constructor arguments:
>
> ```python
> oai = OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")   # local
> oai = OpenAI()                                                         # hosted, key from env
> ```
>
> Everything below that line is identical. For teaching or for CI that can't depend on a
> GPU box, that's a genuinely useful property.

In [ ]:
# The models endpoint works too, so generic tooling can enumerate what you have.
for m in oai.models.list().data:
    print(m.id)

---
# 3. Raw HTTP

No client library at all — just POST JSON. Worth understanding because it's what both
libraries above are doing, it's what the Ollama docs show as `curl` examples, and it's
what you'd port to any other language.

Two endpoints matter: `/api/chat` (message list) and `/api/generate` (single prompt).

In [ ]:
# 3.1 Non-streaming POST to /api/chat
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "What year was the Motorola 68000 released?"}],
    "think": True,
    "stream": False,                 # default is True, so this must be explicit
    "options": {"temperature": 0.6, "num_predict": 1000},
}

r = requests.post(f"{HOST}/api/chat", json=payload, timeout=300)
r.raise_for_status()
data = r.json()

print("thinking:", (data["message"].get("thinking") or "")[:200], "...\n")
print("content :", data["message"]["content"].strip())
print("\ntop-level keys:", ", ".join(data.keys()))

In [ ]:
# 3.2 Streaming: newline-delimited JSON, one object per token-ish chunk
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Count from 1 to 10, comma separated."}],
    "think": False,
    "stream": True,
}

with requests.post(f"{HOST}/api/chat", json=payload, stream=True, timeout=300) as r:
    r.raise_for_status()
    final = None
    for line in r.iter_lines():
        if not line:
            continue
        obj = json.loads(line)
        sys.stdout.write(obj["message"]["content"])
        sys.stdout.flush()
        if obj.get("done"):
            final = obj

print("\n\n--- final chunk carries the metrics ---")
print(f"done_reason : {final['done_reason']}")
print(f"eval_count  : {final['eval_count']}")
print(f"load_duration: {final['load_duration'] / 1e6:.0f} ms")

In the following example, the model uses <think> tags to dileneate the thinking phase from the answer phase.

In [ ]:
import re
def split_thinking(text, open_tag='<think>', close_tag='</think>'):
    """Split a raw completion into (thinking, answer).

    Handles the four shapes that actually occur:
      <think>...</think>answer   both tags present
      ...</think>answer          template opened the block, model never emitted it
      <think>...                 generation stopped mid-thought
      answer                     no reasoning block at all
    """
    # Some chat templates put the opening tag in the prompt, so the returned
    # text begins *inside* the block. Normalise that case first.
    if close_tag in text and (open_tag not in text
                              or text.index(close_tag) < text.index(open_tag)):
        text = open_tag + text

    block = re.compile(re.escape(open_tag) + r'(.*?)' + re.escape(close_tag), re.DOTALL)
    thoughts = [m.group(1) for m in block.finditer(text)]
    answer = block.sub('', text)

    # An unclosed block means the token budget ran out before the model finished.
    if open_tag in answer:
        answer, _, tail = answer.partition(open_tag)
        thoughts.append(tail)

    return '\n\n'.join(t.strip() for t in thoughts).strip(), answer.strip()

In [ ]:
# 3.3 /api/generate — no chat template, no message list.
# raw=True bypasses the template entirely, which is how you'd drive a base model
# or hand-build R1's <｜User｜> / <｜Assistant｜> markers yourself.
payload = {
    "model": MODEL,
    "prompt": "The three classic Unix file permissions are",
    "think": False,
    "stream": False,
    "options": {"num_predict": 1000, "temperature": 0.6},
}

data = requests.post(f"{HOST}/api/generate", json=payload, timeout=300).json()
wrap(split_thinking(data["response"])[1])

In [ ]:
# 3.4 The rest of the surface, for completeness
endpoints = {
    "GET  /api/tags":    lambda: requests.get(f"{HOST}/api/tags").json(),
    "GET  /api/ps":      lambda: requests.get(f"{HOST}/api/ps").json(),
    "POST /api/show":    lambda: requests.post(f"{HOST}/api/show", json={"model": MODEL}).json(),
    "GET  /api/version": lambda: requests.get(f"{HOST}/api/version").json(),
}

for name, fn in endpoints.items():
    out = fn()
    keys = list(out.keys()) if isinstance(out, dict) else type(out).__name__
    print(f"{name:20s} -> {keys}")

### 3.5 When raw HTTP is the right call

- **Unloading on demand** — `{"model": M, "keep_alive": 0}` with an empty prompt evicts
  the model from VRAM. Useful at the end of a batch job.
- **Timeout and retry control** — you own the `requests`/`httpx` session, connection pool
  and backoff, rather than inheriting a library's defaults.
- **Non-Python callers** — the payloads above translate directly to `curl`, Go, or a
  shell script in a marking harness.

In [ ]:
# Evict the model from memory (run this when you're done with the notebook)
# requests.post(f"{HOST}/api/generate", json={"model": MODEL, "keep_alive": 0})

---
# 4. Reasoning-model caveats specific to `deepseek-r1`

These trip people up regardless of which interface they picked.

**No system prompt.** DeepSeek's guidance for R1 is to put all instructions in the *user*
message. A system prompt measurably degrades its reasoning; the model wasn't trained with
one in the reasoning trace.

**No few-shot prompting.** R1's performance drops with in-context examples. Describe the
task directly and zero-shot instead.

**Temperature 0.6, not 0.** At temperature 0 R1 tends to fall into repetition loops
mid-reasoning. 0.5–0.7 is the recommended band, with `top_p=0.95`.

**Force the reasoning to start.** R1 occasionally skips its chain of thought entirely.
DeepSeek's fix is to make the response begin with `<think>\n`. Ollama's template normally
handles this, but if you're using `raw=True` you own it.

**Budget for reasoning tokens.** They count against `num_predict` and against billing on
any hosted equivalent. A hard maths problem can burn several thousand tokens before the
answer starts. Always set `num_predict` in anything automated.

**Raise `num_ctx`.** The 4096 default truncates silently, and reasoning eats context fast.

**Tool calling is unreliable.** Base R1 wasn't trained for function calling. The distill
variants inherit their base model's tool support unevenly, and the reasoning trace tends
to interleave with the tool-call JSON. If you need tools, use a tool-trained model
(`qwen3`, `llama3.1`, `mistral-nemo`) and reserve R1 for reasoning-only steps.

**Drop prior reasoning from history.** As in §1.4 — append `content` to the conversation,
never `thinking`.

In [ ]:
# Demonstrating the num_predict guard: a deliberately hard prompt, capped.
r = chat(
    model=MODEL,
    messages=[{"role": "user",
               "content": "Prove that the square root of 2 is irrational. "
                          "Reason step by step and put the final conclusion on its own line."}],
    think=True,
    options={"temperature": 0.6, "top_p": 0.95, "num_ctx": 16384, "num_predict": 400},
)

think_tokens = len((r.message.thinking or "").split())
print(f"~reasoning words : {think_tokens}")
print(f"total out tokens : {r.eval_count}")
print(f"stopped because  : {r.done_reason}   <- 'length' means the cap was hit\n")
wrap(r.message.content)

---
# 5. Choosing between the three

**Use the `ollama` package** when the model is the core.  `ollama.pull()` / `ollama.show()` let a script guarantee its own prerequisites.

**Use the OpenAI shim** when the *framework* is more important because you're feeding LangChain or an
agent library, or you want one codebase that runs locally in development and against a
hosted model in production. 

**Use raw HTTP** when you're writing infrastructure: a batch runner, a marking harness, a
service with its own retry policy, or anything that has to be portable to another
language.
